# Pneumonia Detection — Inference Pipeline

**Project:** Pneumonia Detection using Deep Learning  
**Model:** MobileNetV2 Transfer Learning  
**Objective:** Load the trained model and run predictions on new chest X-ray images.

This notebook applies the same preprocessing pipeline used during training and outputs a NORMAL or PNEUMONIA classification with confidence score.

## Table of Contents
1. **Configuration & Model Loading** — Load saved model and set threshold
2. **Preprocessing Function** — Same pipeline as training (bicubic, median, CLAHE, z-score)
3. **Prediction Function** — Run inference on any image or .npy file
4. **Visualisation** — Display image with prediction overlay
5. **Interactive Inference** — Enter any image path to get a prediction

## 1. Configuration & Model Loading

We load the trained MobileNetV2 model saved during training. The model file is `xray_mobilenet_model.keras` in the same folder as this notebook.

**Decision threshold:** `0.50` is the default. Raise it to reduce false PNEUMONIA predictions (better NORMAL recall). Lower it to catch more PNEUMONIA cases. `0.70` was used as the final operating threshold after threshold sweep analysis.

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
import matplotlib.pyplot as plt

# CONFIGURATION
SCRIPT_DIR = Path(".")
MODEL_PATH = str(SCRIPT_DIR / 'xray_mobilenet_model.keras')
IMG_SIZE   = (128, 128)

DECISION_THRESHOLD = 0.70

class_names = ['NORMAL', 'PNEUMONIA']

print(f"Model path : {MODEL_PATH}")
print(f"Threshold  : {DECISION_THRESHOLD}")

In [ ]:
print(f"[*] Loading model from {MODEL_PATH}...")
try:
    MODEL = keras.models.load_model(MODEL_PATH)
    print("[+] Model loaded successfully!")
    MODEL.summary()
except Exception as e:
    print(f"[-] Error loading model: {e}")
    MODEL = None

## 2. Preprocessing Function

The exact same five-step pipeline used during training must be applied at inference time. If any step is skipped or changed, the model receives input with a different distribution than what it was trained on, and predictions will be unreliable.

```
Original Image
    |
[1] Read as Grayscale
    |
[2] Bicubic Resize  →  128×128
    |
[3] Median Blur     →  kernel=3  (noise removal)
    |
[4] CLAHE           →  clipLimit=2.0, tileGridSize=(8,8)  (contrast)
    |
[5] Z-Score         →  per-image mean=0, std=1
    |
[6] Add batch + channel dims  →  shape (1, 128, 128, 1)
```

The function also accepts `.npy` files (already preprocessed) directly — useful for running inference on the processed test set.

In [ ]:
def preprocess_single_image(img_path, target_size=IMG_SIZE):
    """
    Standard preprocessing pipeline — must match training exactly.
    Steps: Grayscale read -> Bicubic resize -> Median blur -> CLAHE -> Z-score
    Returns array of shape (1, 128, 128, 1) ready for model.predict()
    """
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not read image at: {img_path}")

    # Cubic resize
    img = cv2.resize(img, target_size, interpolation=cv2.INTER_CUBIC)

    # Median blur — edge-preserving noise removal
    img = cv2.medianBlur(img, 3)

    #  CLAHE — local contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img   = clahe.apply(img)

    # Z-score normalization — makes brightness scanner-independent
    img  = img.astype(np.float32)
    mean = np.mean(img)
    std  = np.std(img)
    if std == 0:
        std = 1
    img = (img - mean) / std

    # Add batch and channel dimensions: (128,128) -> (1, 128, 128, 1)
    img = img[np.newaxis, ..., np.newaxis]
    return img

## 3. Prediction Function

The prediction function handles two input types:
- **Raw image** (`.jpg`, `.jpeg`, `.png`) — runs the full preprocessing pipeline
- **Preprocessed .npy file** — loads directly without reprocessing

It prints preprocessing stats (mean and std) as a quick sanity check. If mean is far from 0 or std is far from 1, the preprocessing may not have been applied correctly.

In [ ]:
def predict_pneumonia(image_path):
    """
    Runs inference on a single image.
    Accepts raw images (.jpg/.png) or preprocessed .npy files.
    Returns: (prediction_label, probability)
    """
    if MODEL is None:
        return "Model not loaded", 0.0

    try:
        if str(image_path).lower().endswith('.npy'):
            # already preprocessed — load directly
            print(f"[*] Loading preprocessed .npy file: {image_path}")
            processed_img = np.load(image_path).astype(np.float32)
            # ensure correct shape
            if len(processed_img.shape) == 2:
                processed_img = processed_img[np.newaxis, ..., np.newaxis]
            elif len(processed_img.shape) == 3:
                processed_img = processed_img[np.newaxis, ...]
        else:
            # raw image — apply full pipeline
            print(f"[*] Preprocessing raw image: {image_path}")
            processed_img = preprocess_single_image(image_path)

        # sanity check — mean ~0, std ~1 if preprocessing correct
        print(f"[*] Preprocessing stats -> Mean: {np.mean(processed_img):.4f}, Std: {np.std(processed_img):.4f}")

        # predict — output is [[probability]]
        prob  = float(MODEL.predict(processed_img, verbose=0)[0][0])
        label = 'PNEUMONIA' if prob > DECISION_THRESHOLD else 'NORMAL'

        return label, prob

    except Exception as e:
        import traceback
        traceback.print_exc()
        return f"Error: {str(e)}", 0.0

## 4. Visualisation

Displays the image with its prediction and confidence score. Saves the visualisation to `training_logs/` alongside the training output files.

In [ ]:
def visualize_prediction(image_path, prediction, probability):
    """
    Displays and saves the image with prediction overlay.
    Green title = NORMAL, Red title = PNEUMONIA.
    """
    try:
        if str(image_path).lower().endswith('.npy'):
            img_data = np.load(image_path)
            img_vis  = ((img_data - img_data.min()) / (img_data.max() - img_data.min()) * 255).astype(np.uint8)
        else:
            img_vis = cv2.imread(str(image_path))
            img_vis = cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(6, 6))
        plt.imshow(img_vis, cmap='gray' if len(img_vis.shape) == 2 else None)

        color = 'red' if prediction == 'PNEUMONIA' else 'green'
        title = f"Prediction: {prediction}\nProbability: {probability:.4f}  |  Threshold: {DECISION_THRESHOLD}"

        plt.title(title, fontsize=13, color=color, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()

        save_dir = SCRIPT_DIR / 'training_logs'
        save_dir.mkdir(exist_ok=True)
        save_name = f"prediction_{Path(str(image_path)).stem}.png"
        plt.savefig(save_dir / save_name, bbox_inches='tight', dpi=150)
        print(f"[+] Saved to: training_logs/{save_name}")

        plt.show()

    except Exception as e:
        print(f"[-] Visualisation error: {e}")

## 5. Run Inference

### Option A - Single Image

Set `IMAGE_PATH` to any chest X-ray image path and run this cell.

In [ ]:
IMAGE_PATH = "../../chest_xray/test/PNEUMONIA/person1685_virus_2903.jpeg"

import platform
if platform.system() == 'Linux' and len(IMAGE_PATH) > 2 and IMAGE_PATH[1] == ':':
    drive = IMAGE_PATH[0].lower()
    IMAGE_PATH = f"/mnt/{drive}/" + IMAGE_PATH[3:].replace("\\", "/")
    print(f"[*] Converted to WSL path: {IMAGE_PATH}")

if os.path.exists(IMAGE_PATH):
    prediction, confidence = predict_pneumonia(IMAGE_PATH)

    print("\n" + "-"*35)
    print(f"  RESULT     : {prediction}")
    print(f"  CONFIDENCE : {confidence:.4f}")
    print(f"  THRESHOLD  : {DECISION_THRESHOLD}")
    print("-"*35)

    visualize_prediction(IMAGE_PATH, prediction, confidence)
else:
    print(f"[-] File not found: {IMAGE_PATH}")

### Option B - Batch Inference on Test Set

Run predictions across all test images and print a summary.

In [ ]:
import os
from pathlib import Path

TEST_DIR    = "../../chest_xray/test_processed/"
correct     = 0
total       = 0
results     = []

for class_name, true_label in [('NORMAL', 0), ('PNEUMONIA', 1)]:
    folder = os.path.join(TEST_DIR, class_name)
    if not os.path.exists(folder):
        print(f"[-] Folder not found: {folder}")
        continue

    files = [f for f in os.listdir(folder) if f.endswith('.npy')]
    print(f"\nRunning on {len(files)} {class_name} images...")

    for fname in files:
        fpath = os.path.join(folder, fname)
        pred_label, prob = predict_pneumonia(fpath)
        pred_int = 1 if pred_label == 'PNEUMONIA' else 0
        is_correct = (pred_int == true_label)
        correct += int(is_correct)
        total   += 1
        results.append({'file': fname, 'true': class_name, 'pred': pred_label,
                        'prob': prob, 'correct': is_correct})

print(f"\n{'-'*40}")
print(f"  Total images  : {total}")
print(f"  Correct       : {correct}")
print(f"  Accuracy      : {correct/total:.4f}" if total > 0 else "  No images found")
print(f"{'-'*40}")

### Option C - Interactive (Enter path manually)

In [ ]:
import platform

print("\n" + "="*40)
print(" PNEUMONIA DETECTION — INTERACTIVE MODE")
print("="*40)
print("Type '1' to exit.\n")

while True:
    user_input = input("Image path: ").strip().replace('"', '').replace("'", "")

    if user_input in ['1', 'q', 'exit', 'quit']:
        print("[*] Done.")
        break

    if not user_input:
        continue

    # converting Windows path to WSL path automatically
    if platform.system() == 'Linux' and len(user_input) > 2 and user_input[1] == ':':
        drive = user_input[0].lower()
        user_input = f"/mnt/{drive}/" + user_input[3:].replace("\\", "/")
        print(f"[*] Converted path: {user_input}")

    if os.path.exists(user_input):
        result, confidence = predict_pneumonia(user_input)

        print("\n" + "-"*35)
        print(f"  RESULT     : {result}")
        print(f"  CONFIDENCE : {confidence:.4f}")
        print(f"  THRESHOLD  : {DECISION_THRESHOLD}")
        print("-"*35 + "\n")

        visualize_prediction(user_input, result, confidence)
    else:
        print(f"[-] File not found: {user_input}")